In [3]:
# -------------------------------------------------------
# CELL 1: Setup — GitHub config and Gold data export
# Purpose: Export Gold tables as JSON and push to GitHub
#          so Streamlit dashboard stays fresh after each run
# -------------------------------------------------------

import json
import base64
import requests
from pyspark.sql import functions as F

# GitHub config — token from Spark environment property
GITHUB_TOKEN = spark.conf.get("spark.github.token")
GITHUB_REPO  = "demonjd2026-afk/globalwatch-fabric"
GITHUB_BRANCH = "main"
DATA_PATH    = "streamlit/data"
API_BASE     = f"https://api.github.com/repos/{GITHUB_REPO}/contents"

HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json",
    "Content-Type": "application/json"
}

print(f"✅ Config loaded")
print(f"   Repo  : {GITHUB_REPO}")
print(f"   Branch: {GITHUB_BRANCH}")
print(f"   Path  : {DATA_PATH}/")

StatementMeta(, 55cabaf4-2fc4-4fd7-846f-18865e9802e5, 7, Finished, Available, Finished, False)

✅ Config loaded
   Repo  : demonjd2026-afk/globalwatch-fabric
   Branch: main
   Path  : streamlit/data/


In [1]:
# -------------------------------------------------------
# CELL 2: Read Gold tables and convert to JSONL strings
# Each table exported as newline-delimited JSON (one row per line)
# -------------------------------------------------------

# fact_readings — key metrics for dashboard
fact_df = spark.sql("""
    SELECT location_id, parameter, value, unit, aqi_category,
           exceeds_who_guideline, reading_ts, reading_date, continent,
           country_sk
    FROM gold_globalwatch.dbo.fact_readings
""")

# dim_country — country metadata
country_df = spark.sql("""
    SELECT country_sk, country_code, country_name, continent
    FROM gold_globalwatch.dbo.dim_country
""")

# dim_station — station locations for map
station_df = spark.sql("""
    SELECT station_sk, location_id, location_name, city,
           country_code, latitude, longitude
    FROM gold_globalwatch.dbo.dim_station
    WHERE active_flag = true
""")

# fact_aqi_predictions — ML model output
pred_df = spark.sql("""
    SELECT location_id, country_sk, pm25, predicted_aqi_class
    FROM gold_globalwatch.dbo.fact_aqi_predictions
""")

# Convert each to JSONL string
def df_to_jsonl(df):
    rows = df.toJSON().collect()
    return "\n".join(rows)

exports = {
    "fact_readings.json":        df_to_jsonl(fact_df),
    "dim_country.json":          df_to_jsonl(country_df),
    "dim_station.json":          df_to_jsonl(station_df),
    "fact_aqi_predictions.json": df_to_jsonl(pred_df),
}

print(f"✅ Data ready for export:")
print(f"   fact_readings        : {fact_df.count()} rows")
print(f"   dim_country          : {country_df.count()} rows")
print(f"   dim_station          : {station_df.count()} rows")
print(f"   fact_aqi_predictions : {pred_df.count()} rows")

StatementMeta(, 55cabaf4-2fc4-4fd7-846f-18865e9802e5, 5, Finished, Available, Finished, False)

✅ Data ready for export:
   fact_readings        : 344 rows
   dim_country          : 9 rows
   dim_station          : 121 rows
   fact_aqi_predictions : 74 rows


In [4]:
# -------------------------------------------------------
# CELL 3: Push JSON files to GitHub via REST API
# Strategy: GET existing file SHA → PUT with new content
#           SHA required by GitHub to update existing files
# -------------------------------------------------------

def push_to_github(filename, content_str):
    """Push a file to GitHub, creating or updating as needed"""
    
    file_path = f"{DATA_PATH}/{filename}"
    url = f"{API_BASE}/{file_path}"
    
    # Encode content as base64 (GitHub API requirement)
    content_b64 = base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
    
    # GET existing file SHA (needed for updates)
    get_resp = requests.get(url, headers=HEADERS, params={"ref": GITHUB_BRANCH})
    
    payload = {
        "message": f"data: auto-update {filename} from Fabric pipeline",
        "content": content_b64,
        "branch": GITHUB_BRANCH
    }
    
    if get_resp.status_code == 200:
        # File exists — include SHA for update
        payload["sha"] = get_resp.json()["sha"]
        action = "Updated"
    else:
        action = "Created"
    
    # PUT file
    put_resp = requests.put(url, headers=HEADERS, json=payload)
    
    if put_resp.status_code in [200, 201]:
        print(f"   ✅ {action}: {file_path}")
        return True
    else:
        print(f"   ❌ Failed {filename}: {put_resp.status_code} — {put_resp.text[:200]}")
        return False

# Push all 4 files
print("Pushing data files to GitHub...")
print("-" * 50)

results = {}
for filename, content in exports.items():
    results[filename] = push_to_github(filename, content)

print("-" * 50)
success = sum(results.values())
print(f"✅ {success}/{len(results)} files pushed successfully")

StatementMeta(, 55cabaf4-2fc4-4fd7-846f-18865e9802e5, 8, Finished, Available, Finished, False)

Pushing data files to GitHub...
--------------------------------------------------
   ✅ Updated: streamlit/data/fact_readings.json
   ✅ Updated: streamlit/data/dim_country.json
   ✅ Updated: streamlit/data/dim_station.json
   ✅ Updated: streamlit/data/fact_aqi_predictions.json
--------------------------------------------------
✅ 4/4 files pushed successfully


In [5]:
# -------------------------------------------------------
# CELL 4: Verify files are live on GitHub
# Confirms Streamlit will pick up fresh data on next load
# -------------------------------------------------------

print("Verifying files on GitHub...")
print("-" * 50)

for filename in exports.keys():
    url = f"{API_BASE}/{DATA_PATH}/{filename}"
    resp = requests.get(url, headers=HEADERS, params={"ref": GITHUB_BRANCH})
    if resp.status_code == 200:
        info = resp.json()
        size_kb = info["size"] / 1024
        print(f"   ✅ {filename} — {size_kb:.1f} KB")
    else:
        print(f"   ❌ {filename} — not found")

print("-" * 50)
print("✅ Streamlit dashboard data is up to date!")
print(f"   URL: https://globalwatch-fabric-dbhnvwxfvprsadkxtu6y76.streamlit.app")

StatementMeta(, 55cabaf4-2fc4-4fd7-846f-18865e9802e5, 9, Finished, Available, Finished, False)

Verifying files on GitHub...
--------------------------------------------------
   ✅ fact_readings.json — 64.4 KB
   ✅ dim_country.json — 0.8 KB
   ✅ dim_station.json — 20.2 KB
   ✅ fact_aqi_predictions.json — 6.2 KB
--------------------------------------------------
✅ Streamlit dashboard data is up to date!
   URL: https://globalwatch-fabric-dbhnvwxfvprsadkxtu6y76.streamlit.app
